In [12]:
from astropy.io import fits
import numpy as np
from reproject import reproject_interp
from tqdm import tqdm
import matplotlib.pyplot as plt
from astropy.wcs import WCS
import warnings
from astropy.wcs import FITSFixedWarning
warnings.simplefilter('ignore', FITSFixedWarning)

In [13]:
path_image_1 = "/home/wesley/Desktop/JWST_REU/Jorgensen_2024/"
path_image_2 = "/home/wesley/Desktop/JWST_REU/HEFE_2025/"

In [14]:
hdu_1 = fits.open((path_image_1 + 'Jorgensen_processed_2024_f212n.fits'))[1]
hdu_2= fits.open((path_image_2 + 'hefe_2025_F212N.fits'))[1]
# hdu_2 = fits.open((path_image_1 + 'nan_rm_Jorgensen_2024_f212n_pipeline.fits'))[1]


In [15]:
hdul_1 = fits.open((path_image_1 + 'Jorgensen_processed_2024_f212n.fits'))
hdul_2= fits.open((path_image_2 + 'hefe_2025_F212N.fits'))
# hdul_2 = fits.open((path_image_1 + 'nan_rm_Jorgensen_2024_f212n_pipeline.fits'))


In [ ]:
hdu_1.header['CRPIX1'] += -6.346153846153846
hdu_1.header['CRPIX2'] += 7.684999999999999


**132 days**
2024-10-17 to
2025-02-26 

In [19]:
array_new = reproject_interp(hdu_2, hdu_1.header)

In [20]:
primary = fits.PrimaryHDU()
primary.header = hdul_2[0].header

In [21]:
image = fits.ImageHDU(data=array_new[0])
image.header["EXTNAME"] = "SCI"

wcs = WCS(hdul_1[1].header)
image.header.update(wcs.to_header())


hdul = fits.HDUList([primary, image])
hdul.writeto(path_image_2 + "repro_processed_hefe_2025_F212N.fits", overwrite=True)

In [ ]:
hdul_3 = fits.open((path_image_2 + '3_pix_repro_processed_hefe_2025_F212Ntest.fits'))

In [ ]:
nan_y = np.where(np.isnan(hdul_3[1].data))[0]
nan_x = np.where(np.isnan(hdul_3[1].data))[1]

In [ ]:
for z in tqdm(range(len(nan_x))):
    """
    This takes in the location of all the nan pixels and will find the value of the 8 pixels surrounding it 
    The mean of the 8 pixels is taken as long as the at least 4 pixels have a real number value
    This preforms a check on each pixel to make sure that the pixel is within the bounds of the data and makes it a nan if not
    It then changes the value of the nan pixel to the mean and saves it to a new fits file
    """
    cx, cy = nan_x[z], nan_y[z]
    hdul_1[1].data[nan_y[z], nan_x[z]] = np.nan
hdul_1.writeto(path_image_1 +'sliced_nan_rm_recentered_Jorgensen_processed_2024_f212n.fits', overwrite=True)

In [ ]:
fig, ax = plt.subplots()
ax.imshow(hdu_1.data, vmin=0, vmax=30, origin="lower")
print()

In [ ]:
fig, ax = plt.subplots()
ax.imshow(hdul_1[1].data, vmin=0, vmax=30, origin="lower")
print()

In [ ]:
new_array = hdul_3[1].data - hdu_1.data

In [ ]:
primary = fits.PrimaryHDU()
primary.header = hdul_2[0].header

In [ ]:
from astropy.io import fits
hdu1 = fits.ImageHDU(data=new_array)
hdu1.header["EXTNAME"] = "SCI"

wcs = WCS(hdul_1[1].header)
hdu1.header.update(wcs.to_header())


hdul = fits.HDUList([primary, hdu1])
hdu1.writeto("sliced_Jorgensen_processed_2024_f212n.fits", overwrite=True)

In [ ]:
fig, ax = plt.subplots()
ax.imshow(hdul_3[1].data - hdu_1.data, vmin=0, vmax=30, origin="lower")
print()